In [12]:
import numpy as np
import xarray as xr
import dask.array as da
from virtualizarr import open_virtual_dataset
import pandas as pd

In [13]:
# Create example data with Dask arrays
time = pd.date_range(start="2000-01-01", end="2003-12-31", freq="D")
x = np.arange(5)  # 5 x points
y = np.arange(3)  # 3 y points

# Create dask array with time chunked to 1
data = da.random.random((len(time), len(x), len(y)))

# Create xarray Dataset with dask arrays
ds = xr.Dataset(
    {
        "temperature": (["time", "x", "y"], data),
    },
    coords={
        "time": time,
        "x": x,
        "y": y,
    },
)

print(ds)

<xarray.Dataset> Size: 187kB
Dimensions:      (time: 1461, x: 5, y: 3)
Coordinates:
  * time         (time) datetime64[us] 12kB 2000-01-01 2000-01-02 ... 2003-12-31
  * x            (x) int64 40B 0 1 2 3 4
  * y            (y) int64 24B 0 1 2
Data variables:
    temperature  (time, x, y) float64 175kB dask.array<chunksize=(1461, 5, 3), meta=np.ndarray>


In [6]:
output_path = "example_data.zarr"

In [7]:
# Define chunking: time=1, x=50, y=30 (full x and y, but time chunked by 1)
ds.to_zarr(output_path, mode="w", encoding={"temperature": {"chunks": (1, 50, 30)}})

print(f"Dataset saved to {output_path}")
print("Chunking: time=1, x=50, y=30")

/Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Dataset saved to example_data.zarr
Chunking: time=1, x=50, y=30


In [9]:
ds_averaged = ds.groupby("time.dayofyear").mean("time")
ds_averaged

<xarray.Dataset> Size: 47kB
Dimensions:      (dayofyear: 366, x: 5, y: 3)
Coordinates:
  * dayofyear    (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
  * x            (x) int64 40B 0 1 2 3 4
  * y            (y) int64 24B 0 1 2
Data variables:
    temperature  (dayofyear, x, y) float64 44kB dask.array<chunksize=(1, 5, 3), meta=np.ndarray>

In [10]:
ds_averaged.to_zarr(
    "example_data_averaged.zarr",
    mode="w",
    encoding={"temperature": {"chunks": (1, 50, 30)}},
)

/Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [5]:
# Verify the chunking
ds_loaded = xr.open_zarr(output_path)
print("\nLoaded dataset:")
print(ds_loaded)
print("\nChunk information:")
print(ds_loaded.temperature.chunks)


Loaded dataset:
<xarray.Dataset> Size: 121kB
Dimensions:      (time: 10, x: 50, y: 30)
Coordinates:
  * time         (time) int64 80B 0 1 2 3 4 5 6 7 8 9
  * x            (x) int64 400B 0 1 2 3 4 5 6 7 8 ... 41 42 43 44 45 46 47 48 49
  * y            (y) int64 240B 0 1 2 3 4 5 6 7 8 ... 21 22 23 24 25 26 27 28 29
Data variables:
    temperature  (time, x, y) float64 120kB dask.array<chunksize=(1, 50, 30), meta=np.ndarray>

Chunk information:
((1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (50,), (30,))


In [14]:
# Fix for asyncio event loop issue in Jupyter
import nest_asyncio

nest_asyncio.apply()

In [15]:
from virtualizarr.parsers import ZarrParser
from obstore.store import LocalStore
from virtualizarr.registry import ObjectStoreRegistry

zarr_store = "/Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/src/" + str(
    output_path
)
store = LocalStore(prefix=zarr_store)
registry = ObjectStoreRegistry({f"file://{zarr_store}": store})
parser = ZarrParser()
vds = open_virtual_dataset(url=zarr_store, registry=registry, parser=parser)

In [5]:
vds

<xarray.Dataset> Size: 121kB
Dimensions:      (time: 10, x: 50, y: 30)
Coordinates:
  * time         (time) int64 80B 0 1 2 3 4 5 6 7 8 9
  * x            (x) int64 400B 0 1 2 3 4 5 6 7 8 ... 41 42 43 44 45 46 47 48 49
  * y            (y) int64 240B 0 1 2 3 4 5 6 7 8 ... 21 22 23 24 25 26 27 28 29
Data variables:
    temperature  (time, x, y) float64 120kB ManifestArray<shape=(10, 50, 30),...

In [16]:
refs = vds.vz.to_kerchunk()

In [12]:
refs

{'version': 1,
 'refs': {'.zgroup': '{"zarr_format":2}',
  '.zattrs': '{}',
  'time/0': 'base64:AAAAAAAAAAABAAAAAAAAAAIAAAAAAAAAAwAAAAAAAAAEAAAAAAAAAAUAAAAAAAAABgAAAAAAAAAHAAAAAAAAAAgAAAAAAAAACQAAAAAAAAA=',
  'time/.zarray': '{"shape":[10],"chunks":[10],"dtype":"<i8","fill_value":null,"order":"C","filters":null,"dimension_separator":".","compressor":null,"attributes":{},"zarr_format":2}',
  'time/.zattrs': '{"_ARRAY_DIMENSIONS":["time"]}',
  'x/0': 'base64:AAAAAAAAAAABAAAAAAAAAAIAAAAAAAAAAwAAAAAAAAAEAAAAAAAAAAUAAAAAAAAABgAAAAAAAAAHAAAAAAAAAAgAAAAAAAAACQAAAAAAAAAKAAAAAAAAAAsAAAAAAAAADAAAAAAAAAANAAAAAAAAAA4AAAAAAAAADwAAAAAAAAAQAAAAAAAAABEAAAAAAAAAEgAAAAAAAAATAAAAAAAAABQAAAAAAAAAFQAAAAAAAAAWAAAAAAAAABcAAAAAAAAAGAAAAAAAAAAZAAAAAAAAABoAAAAAAAAAGwAAAAAAAAAcAAAAAAAAAB0AAAAAAAAAHgAAAAAAAAAfAAAAAAAAACAAAAAAAAAAIQAAAAAAAAAiAAAAAAAAACMAAAAAAAAAJAAAAAAAAAAlAAAAAAAAACYAAAAAAAAAJwAAAAAAAAAoAAAAAAAAACkAAAAAAAAAKgAAAAAAAAArAAAAAAAAACwAAAAAAAAALQAAAAAAAAAuAAAAAAAAAC8AAAAAAAAAMAAAAAAAAAAxAAAAAAAAAA==',


In [17]:
# Load the averaged dataset to get its references
zarr_store_avg = (
    "/Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/src/example_data_averaged.zarr"
)
store_avg = LocalStore(prefix=zarr_store_avg)
registry_avg = ObjectStoreRegistry({f"file://{zarr_store_avg}": store_avg})
vds_avg = open_virtual_dataset(url=zarr_store_avg, registry=registry_avg, parser=parser)
refs_avg = vds_avg.vz.to_kerchunk()

print("Averaged dataset refs loaded")
print(
    f"Number of dayofyear chunks: {len([k for k in refs_avg['refs'].keys() if k.startswith('temperature/') and '.0.0' in k])}"
)

Averaged dataset refs loaded
Number of dayofyear chunks: 366


In [18]:
import json
import base64

# Create new refs dict based on ds time dimension but referencing ds_averaged chunks
new_refs = {"version": 1, "refs": {}}

# Copy basic metadata
new_refs["refs"][".zgroup"] = refs["refs"][".zgroup"]
new_refs["refs"][".zattrs"] = refs["refs"][".zattrs"]

# Copy x and y (spatial dimensions remain the same)
for key in refs["refs"].keys():
    if key.startswith("x/") or key.startswith("y/"):
        new_refs["refs"][key] = refs["refs"][key]

# Get the original time coordinate from ds
time_array_metadata = json.loads(refs["refs"]["time/.zarray"])
n_times = time_array_metadata["shape"][0]  # Total number of time steps in ds

# Get day of year for each time step
time_coords = ds.time.values
dayofyear = pd.DatetimeIndex(time_coords).dayofyear

print(f"Total time steps in ds: {n_times}")
print(f"Day of year range: {dayofyear.min()} to {dayofyear.max()}")
print(f"Number of unique day of years: {len(np.unique(dayofyear))}")

Total time steps in ds: 1461
Day of year range: 1 to 366
Number of unique day of years: 366


In [24]:
# Keep the original time coordinate from ds (full date range)
new_refs["refs"]["time/0"] = refs["refs"]["time/0"]
new_refs["refs"]["time/.zarray"] = refs["refs"]["time/.zarray"]
new_refs["refs"]["time/.zattrs"] = refs["refs"]["time/.zattrs"]

# Map temperature chunks: each time step references the corresponding dayofyear chunk
for time_idx in range(n_times):
    # Get the day of year for this time step (1-indexed in pandas, but 0-indexed in zarr)
    doy = dayofyear[time_idx]

    # dayofyear is 1-indexed (1-366), but zarr chunks are 0-indexed
    # So doy 1 maps to chunk 0, doy 2 maps to chunk 1, etc.
    doy_chunk_idx = doy - 1

    # Reference the corresponding dayofyear chunk from ds_averaged
    new_key = f"temperature/{time_idx}.0.0"
    avg_key = f"temperature/{doy_chunk_idx}.0.0"

    if avg_key in refs_avg["refs"]:
        new_refs["refs"][new_key] = refs_avg["refs"][avg_key]
    else:
        print(f"Warning: {avg_key} not found in averaged refs")

# Update temperature metadata with new time dimension size
temp_array_metadata = json.loads(refs_avg["refs"]["temperature/.zarray"])
temp_array_metadata["shape"] = [
    n_times,
    temp_array_metadata["shape"][1],
    temp_array_metadata["shape"][2],
]

new_refs["refs"]["temperature/.zarray"] = json.dumps(temp_array_metadata)

# Update temperature attributes to use 'time' dimension instead of 'dayofyear'
temp_attrs = json.loads(refs_avg["refs"]["temperature/.zattrs"])
temp_attrs["_ARRAY_DIMENSIONS"] = ["time", "x", "y"]  # Rename dayofyear to time
new_refs["refs"]["temperature/.zattrs"] = json.dumps(temp_attrs)

print("\nNew refs created:")
print(f"  Time dimension: {n_times} steps (full date range)")
print(f"  Temperature shape: {temp_array_metadata['shape']}")
print("  Temperature chunks mapped from ds_averaged based on day of year")


New refs created:
  Time dimension: 1461 steps (full date range)
  Temperature shape: [1461, 5, 3]
  Temperature chunks mapped from ds_averaged based on day of year


In [25]:
# Write the new refs to JSON
refs_json_path = (
    "/Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/src/climatology_refs.json"
)

with open(refs_json_path, "w") as f:
    json.dump(new_refs, f, indent=2)

print(f"Climatology refs written to: {refs_json_path}")

Climatology refs written to: /Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/src/climatology_refs.json


In [26]:
# Open the climatology dataset with xarray
import fsspec

fs = fsspec.filesystem("reference", fo=refs_json_path)
mapper = fs.get_mapper("")

ds_climatology = xr.open_dataset(
    mapper, engine="zarr", backend_kwargs={"consolidated": False}
)
ds_climatology

<xarray.Dataset> Size: 187kB
Dimensions:      (time: 1461, x: 5, y: 3)
Coordinates:
  * time         (time) datetime64[ns] 12kB 2000-01-01 2000-01-02 ... 2003-12-31
  * x            (x) int64 40B 0 1 2 3 4
  * y            (y) int64 24B 0 1 2
Data variables:
    temperature  (time, x, y) float64 175kB ...

In [27]:
# Check the actual structure
print("ds_climatology dimensions:", ds_climatology.dims)
print("ds_climatology coords:", list(ds_climatology.coords.keys()))
print("\nds_averaged dimensions:", ds_averaged.dims)
print("ds_averaged coords:", list(ds_averaged.coords.keys()))

ds_climatology dimensions: FrozenMappingWarningOnValuesAccess({'time': 1461, 'x': 5, 'y': 3})
ds_climatology coords: ['x', 'y', 'time']

ds_averaged dimensions: FrozenMappingWarningOnValuesAccess({'dayofyear': 366, 'x': 5, 'y': 3})
ds_averaged coords: ['x', 'y', 'dayofyear']


In [28]:
# Verify: Check that same day of year across different years has the same values
jan_1_2000 = ds_climatology.temperature.sel(time="2000-01-01").values
jan_1_2001 = ds_climatology.temperature.sel(time="2001-01-01").values
jan_1_2002 = ds_climatology.temperature.sel(time="2002-01-01").values

print("Verification: Same day of year should have same values across years")
print(f"Jan 1, 2000 equals Jan 1, 2001: {np.allclose(jan_1_2000, jan_1_2001)}")
print(f"Jan 1, 2000 equals Jan 1, 2002: {np.allclose(jan_1_2000, jan_1_2002)}")

# Check different day
feb_29_2000 = ds_climatology.temperature.sel(time="2000-02-29").values  # leap year
feb_29_2004 = (
    ds_climatology.temperature.sel(time="2004-02-29")
    if "2004-02-29" in ds.time.dt.strftime("%Y-%m-%d").values
    else None
)
if feb_29_2004 is not None:
    print("Feb 29, 2000 would equal Feb 29, 2004: both are day 60")

print(
    "\n✓ Climatology mapping confirmed: Each date references its corresponding day-of-year from ds_averaged!"
)

Verification: Same day of year should have same values across years
Jan 1, 2000 equals Jan 1, 2001: True
Jan 1, 2000 equals Jan 1, 2002: True

✓ Climatology mapping confirmed: Each date references its corresponding day-of-year from ds_averaged!


In [15]:
import json
import numpy as np

# Define how many times to repeat along time dimension
n_repeats = 3

# Get the original time array
time_array_metadata = json.loads(refs["refs"]["time/.zarray"])
original_time_shape = time_array_metadata["shape"][0]

# Decode the original time data (remove 'base64:' prefix)
time_data_b64 = refs["refs"]["time/0"].replace("base64:", "")
time_data = base64.b64decode(time_data_b64)
original_time = np.frombuffer(time_data, dtype="<i8")

# Repeat the time array
repeated_time = np.tile(original_time, n_repeats)

# Encode back to base64
repeated_time_base64 = "base64:" + base64.b64encode(repeated_time.tobytes()).decode(
    "ascii"
)

# Update time array metadata
time_array_metadata["shape"] = [len(repeated_time)]
time_array_metadata["chunks"] = [len(repeated_time)]

# Update refs for time
refs["refs"]["time/0"] = repeated_time_base64
refs["refs"]["time/.zarray"] = json.dumps(time_array_metadata)

# Get the temperature array metadata
temp_array_metadata = json.loads(refs["refs"]["temperature/.zarray"])
original_temp_shape = temp_array_metadata["shape"][0]

# Update temperature shape
temp_array_metadata["shape"] = [
    original_temp_shape * n_repeats,
    temp_array_metadata["shape"][1],
    temp_array_metadata["shape"][2],
]

# Create repeated references for temperature chunks
for repeat_idx in range(n_repeats):
    for time_idx in range(original_temp_shape):
        new_time_idx = repeat_idx * original_temp_shape + time_idx
        original_key = f"temperature/{time_idx}.0.0"
        new_key = f"temperature/{new_time_idx}.0.0"

        # Reference the same original chunk
        refs["refs"][new_key] = refs["refs"][original_key]

# Update temperature array metadata
refs["refs"]["temperature/.zarray"] = json.dumps(temp_array_metadata)

print(f"Original time shape: {original_time_shape}")
print(f"New time shape: {len(repeated_time)}")
print("Original temperature shape: [10, 50, 30]")
print(f"New temperature shape: {temp_array_metadata['shape']}")

Original time shape: 10
New time shape: 30
Original temperature shape: [10, 50, 30]
New temperature shape: [30, 50, 30]


In [17]:
# Write refs dict to JSON file
refs_json_path = (
    "/Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/src/repeated_refs.json"
)

with open(refs_json_path, "w") as f:
    json.dump(refs, f, indent=2)

print(f"Refs dict written to: {refs_json_path}")

Refs dict written to: /Users/haukeschulz/Documents/GitHub/FBC_Lagrangian/src/repeated_refs.json


In [19]:
# Open with xarray using kerchunk and fsspec
import xarray as xr

# Open with xarray
ds_repeated = xr.open_dataset(
    "reference://",
    engine="zarr",
    storage_options={"fo": refs_json_path, "consolidated": False},
)
ds_repeated

/var/folders/fj/g0x4n_f15tb6zfwjhzc8gvzr0000gn/T/ipykernel_48938/1544466557.py:5: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds_repeated = xr.open_dataset("reference://", engine="zarr", storage_options={'fo': refs_json_path, 'consolidated': False})


<xarray.Dataset> Size: 361kB
Dimensions:      (time: 30, x: 50, y: 30)
Coordinates:
  * time         (time) int64 240B 0 1 2 3 4 5 6 7 8 9 0 ... 0 1 2 3 4 5 6 7 8 9
  * x            (x) int64 400B 0 1 2 3 4 5 6 7 8 ... 41 42 43 44 45 46 47 48 49
  * y            (y) int64 240B 0 1 2 3 4 5 6 7 8 ... 21 22 23 24 25 26 27 28 29
Data variables:
    temperature  (time, x, y) float64 360kB ...